# Simple CNN for Yaris Detection

## Preprocessing of data

Imports pictures from labeled folders. Pictures are squared end resized before stored into preprocessed folder. 

This part of the notebook only needs to be run once, since the training portion reads the preprocessed data. 

In [ ]:
import os

def get_folder_names(directory):
    folder_names = [name for name in os.listdir(directory) if os.path.isdir(os.path.join(directory, name))]
    return folder_names

# Example usage
directory_path = '../../data/VMMRdb'
folder_names = get_folder_names(directory_path)
print(folder_names[0:10])

In [ ]:
from yaris_pipe import pipe_pictures

image_lists = []
labels = []
length = len(folder_names)
for i, name in enumerate(folder_names):
    path = directory_path+'/'+name
    images = pipe_pictures(path)
    if images is not None:
        image_lists.append(images)
        labels.append(name)
    print(f'{i+1} of {length} done', end='\r')

In [ ]:
import os
import cv2

preprocess_path = '../../data/preprocessed_data'

# Create directories if they don't exist
for label in labels:
    folder_path = os.path.join(preprocess_path, label)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

# Save images to corresponding folders
n_labels = len(image_lists)
for i, image_sublist in enumerate(image_lists):
    label = labels[i]
    for j, image in enumerate(image_sublist):
        folder_path = os.path.join(preprocess_path, label)
        image_path = os.path.join(folder_path, f'image_{label}_{j}.jpg')
        try:
            cv2.imwrite(image_path, image)
        except:
            print(f'Failed for {label} {j}')
    print(f"Finished {label}, {i+1} of {n_labels}")

Finished with Non-yaris-XP10 data. Moving on to yaris XP10 pictures

In [ ]:
path = '../../data/yaris_downloads_clean'
yaris_images = pipe_pictures(path)
if yaris_images is not None:
    print(f'Found {len(yaris_images)} yaris images')

In [ ]:
# make yaris label
label = 'yaris_XP10'
folder_path = os.path.join(preprocess_path, label)
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

for j, image in enumerate(yaris_images):
    folder_path = os.path.join(preprocess_path, label)
    image_path = os.path.join(folder_path, f'image_{label}_{j}.jpg')
    try:
        cv2.imwrite(image_path, image)
    except:
        print(f'Failed for {label} {j}')
print(f"Finished preprocessing yaris")

## Model training

All code below is model training. The above part is preprocessing and should only be needed to run once in order to preprocess the data. It may take some time to run the preprocessing step


In [1]:
import tensorflow as tf
from keras import layers, models
import numpy as np
import os, cv2

def get_folder_names(directory):
    folder_names = [name for name in os.listdir(directory) if os.path.isdir(os.path.join(directory, name))]
    return folder_names

In [2]:
from yaris_pipe import read_images_from_folder

preprocess_path = '../../data/preprocessed_data'
data_folders = get_folder_names(preprocess_path)
image_lists = []
labels = []
length = len(data_folders)
for i, name in enumerate(data_folders):
    path = preprocess_path+'/'+name
    images = read_images_from_folder(path)
    #if i == 5000: break
    if images is not None:
        image_lists.append(images)
        if name == 'yaris_XP10': labels.append(1)
        else: labels.append(0)
    print(f'{i+1} of {length} done', end='\r')


In [3]:
# # Flatten the list of image lists into a single list of images
all_images = []
for i, image in enumerate(image_lists):
    for img in image:
        all_images.append(np.array(img))
    print(f'{i+1}', end='\r')

print('Finished flattening list')

# Convert the list of images to a numpy array
X = np.array(all_images)

#print('Normalising')
# Normalize the pixel values to be between 0 and 1
#X = X / 255.0
print(f'X shape: {X.shape}')
# Convert labels to a numpy array
all_labels = [label for label, sublist in zip(labels, image_lists) for _ in sublist]
y = np.array(all_labels)
print(f'y shape: {y.shape}')

Finished flattening list
X shape: (285182, 100, 100, 3)
y shape: (285182,)


In [4]:
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(100, 100, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

/Users/peterlawrence/Repos/Personal/Yaris-detector/venv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(2))

In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 98, 98, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 49, 49, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 47, 47, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 23, 23, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 21, 21, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 28224)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,806,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,862,850 (7.11 MB)

 Trainable params: 1,862,850 (7.11 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

history = model.fit(X, y, epochs=10)

Epoch 1/10
2085/8912 ━━━━━━━━━━━━━━━━━━━━ 7:16 64ms/step - accuracy: 0.9985 - loss: 0.4172

KeyboardInterrupt: 